# MedPulse: Gemma 3 + LoRA/PEFT in Google Colab

**Goal:** educational prototype that rewrites fictional post-discharge summaries in plain language.

**Safety:** use synthetic examples only. Do not upload real patient data. This is not medical advice or a clinical device; clinician review is required before any real use. The public Gradio link is temporary and anyone with the link may be able to access the demo while it is running.

Recommended starting model: `google/gemma-3-1b-it`. The 4B model may require more GPU memory and runtime. You must have Hugging Face access to the selected gated Gemma model.


In [ ]:
# 1) Install dependencies (run in a fresh Colab runtime)
%pip install -q "transformers>=4.51.0,<5.0.0" "accelerate>=1.2.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" "datasets>=3.2.0" "gradio>=5.0.0" "sentencepiece>=0.2.0" safetensors


In [ ]:

from google.colab import drive
from huggingface_hub import notebook_login

drive.mount("/content/drive")

# Enter your Hugging Face token in the secure login prompt.
notebook_login()


In [ ]:

from huggingface_hub import HfApi

api = HfApi()
user = api.whoami()

print("Login successful!")
print("Hugging Face username:", user["name"])


In [ ]:

from huggingface_hub import model_info

info = model_info("google/gemma-3-4b-it")
print("Model repository is accessible:", info.id)


In [ ]:
# 3) Create project files in the Colab workspace
from pathlib import Path
import os, json
ROOT = Path("/content/medpulse")
(ROOT / "src").mkdir(parents=True, exist_ok=True)
(ROOT / "data").mkdir(exist_ok=True)
(ROOT / "outputs").mkdir(exist_ok=True)
os.chdir(ROOT)
print("Workspace:", Path.cwd())


In [ ]:
%%writefile /content/medpulse/src/medpulse_data.py
"""Generate a small, fictional-only dataset for MedPulse demonstrations."""
from __future__ import annotations
import json
from pathlib import Path

SYNTHETIC_CASES = [
    {
        "id": "SYN001",
        "summary": "You were treated for dehydration. Drink fluids regularly as advised by your care team. Take the medicines listed on your discharge sheet exactly as prescribed. Arrange a follow-up visit with your clinician in 3 days. Seek urgent medical care if you faint, become confused, cannot keep fluids down, or feel rapidly worse.",
        "plain_language": "You were treated for dehydration, which means your body did not have enough fluid. Drink fluids regularly, following the plan your care team gave you. Take medicines exactly as written on your discharge sheet. See your clinician in 3 days. Get urgent medical help if you faint, feel confused, cannot keep fluids down, or get much worse."
    },
    {
        "id": "SYN002",
        "summary": "You were evaluated for a respiratory infection and discharged home. Rest and drink fluids. Use medications only as directed on your discharge paperwork. Follow up with your clinician within 5 days. Seek urgent care for severe difficulty breathing, blue or grey lips, new confusion, or rapidly worsening symptoms.",
        "plain_language": "You were checked for a breathing-related infection and can continue recovering at home. Rest and drink fluids. Use medicines only as your discharge papers direct. Contact your clinician for a follow-up within 5 days. Get urgent help if breathing becomes very difficult, your lips look blue or grey, you become confused, or you get much worse."
    },
    {
        "id": "SYN003",
        "summary": "You were discharged after treatment for a minor ankle sprain. Follow the activity and elevation instructions given by your care team. Use pain relief only according to your discharge instructions. Arrange follow-up if advised. Seek medical review for increasing pain, marked swelling, numbness, or a cold or discoloured foot.",
        "plain_language": "You were treated for a mild ankle sprain. Follow your care team's instructions about activity and raising your ankle. Take pain medicine only as your discharge instructions say. Attend follow-up if you were asked to. Get medical help if pain or swelling increases a lot, the area becomes numb, or your foot feels cold or changes colour."
    },
    {
        "id": "SYN004",
        "summary": "You were assessed for stomach upset and discharged with home-care instructions. Take prescribed medicines exactly as directed and follow the diet guidance provided. Arrange follow-up as written on your discharge paperwork. Seek urgent medical care for severe or persistent abdominal pain, blood in vomit or stool, fainting, or inability to keep fluids down.",
        "plain_language": "You were checked for stomach problems and sent home with care instructions. Take prescribed medicines exactly as directed and follow the food guidance you were given. Check your discharge papers for your follow-up plan. Get urgent medical help for severe or ongoing belly pain, blood in vomit or stool, fainting, or if you cannot keep fluids down."
    },
    {
        "id": "SYN005",
        "summary": "You were discharged following evaluation for a headache. Follow the clinician's instructions and take medicines only as prescribed. Keep the planned follow-up appointment. Seek emergency care for a sudden extremely severe headache, weakness on one side, difficulty speaking, new confusion, or vision loss.",
        "plain_language": "You were checked for a headache and sent home. Follow your clinician's instructions and take medicines only as prescribed. Keep your planned follow-up appointment. Get emergency help for a sudden, extremely severe headache, weakness on one side, trouble speaking, new confusion, or loss of vision."
    },
    {
        "id": "SYN006",
        "summary": "You were discharged after observation for a minor skin infection. Follow wound-care instructions and complete prescribed medication only as directed. Attend the scheduled follow-up. Seek medical advice promptly if redness spreads, pain or swelling worsens, fever develops, or you feel significantly more unwell.",
        "plain_language": "You were observed for a minor skin infection and can continue care at home. Follow the wound-care steps you were given and take prescribed medicine exactly as directed. Go to your scheduled follow-up. Contact a healthcare professional promptly if redness spreads, pain or swelling gets worse, you develop a fever, or you feel much more unwell."
    },
    {
        "id": "SYN007",
        "summary": "You were evaluated for a urinary complaint and discharged home. Follow the hydration and medication instructions on your discharge sheet. Arrange follow-up as directed. Seek urgent medical care for fever with shaking chills, severe back or side pain, repeated vomiting, or rapidly worsening symptoms.",
        "plain_language": "You were checked for a urinary problem and sent home. Follow the fluid and medicine instructions on your discharge sheet. Arrange follow-up as directed. Get urgent medical help if you have a fever with shaking chills, severe back or side pain, repeated vomiting, or symptoms that get worse quickly."
    },
    {
        "id": "SYN008",
        "summary": "You were discharged after evaluation for elevated blood pressure without an emergency finding. Take any prescribed medicines exactly as directed and record readings if your clinician asked you to. Keep your follow-up appointment. Seek emergency care for chest pain, severe shortness of breath, fainting, or new weakness or difficulty speaking.",
        "plain_language": "Your blood pressure was high, but the assessment did not find an emergency at that time. Take prescribed medicines exactly as directed. Record your blood pressure if your clinician asked you to, and keep your follow-up appointment. Get emergency help for chest pain, severe trouble breathing, fainting, new weakness, or difficulty speaking."
    },
]

def write_jsonl(path: str | Path = "data/medpulse_synthetic.jsonl") -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for case in SYNTHETIC_CASES:
            record = {
                "id": case["id"],
                "source_type": "synthetic",
                "instruction": (
                    "Rewrite the discharge summary in plain, calm language. "
                    "Preserve all dates, medicine names and doses, follow-up instructions, "
                    "warning signs, and emergency directions. Do not invent medical advice. "
                    "If information is missing, say to check with the care team."
                ),
                "input": case["summary"],
                "response": case["plain_language"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return path

def validate_jsonl(path: str | Path = "data/medpulse_synthetic.jsonl") -> int:
    count = 0
    with Path(path).open(encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            row = json.loads(line)
            for key in ("instruction", "input", "response", "source_type"):
                if not row.get(key):
                    raise ValueError(f"Line {line_number}: missing {key}")
            if row["source_type"] != "synthetic":
                raise ValueError(f"Line {line_number}: non-synthetic record rejected")
            count += 1
    if count == 0:
        raise ValueError("Dataset is empty")
    return count

if __name__ == "__main__":
    output = write_jsonl()
    print(f"Wrote {validate_jsonl(output)} synthetic examples to {output}")


In [ ]:
%%writefile /content/medpulse/src/medpulse_train.py
"""Minimal PEFT LoRA training helpers for a text-only Gemma 3 instruction model."""
from __future__ import annotations
import json
from pathlib import Path
from typing import Any

def load_records(path: str | Path) -> list[dict[str, Any]]:
    records = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
    if not records:
        raise ValueError("No training records found")
    if any(r.get("source_type") != "synthetic" for r in records):
        raise ValueError("This starter accepts synthetic examples only.")
    return records

def format_example(tokenizer, row: dict[str, str], max_length: int = 1024):
    """Tokenize prompt + answer and mask prompt tokens from the loss."""
    prompt = (
        "You are MedPulse, an educational assistant that rewrites discharge summaries "
        "in plain language. Preserve medicine names/doses, follow-up instructions, warning "
        "signs, and emergency directions. Do not add advice.\n\n"
        f"Instruction: {row['instruction']}\n"
        f"Discharge summary: {row['input']}\n"
        "Plain-language summary:"
    )
    answer = row["response"] + (tokenizer.eos_token or "")
    prompt_ids = tokenizer(prompt, add_special_tokens=True, truncation=True, max_length=max_length)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False, truncation=True, max_length=max_length)["input_ids"]
    input_ids = (prompt_ids + answer_ids)[:max_length]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:max_length]
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

class PadCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, features):
        import torch
        max_len = max(len(x["input_ids"]) for x in features)
        pad_id = self.tokenizer.pad_token_id
        if pad_id is None:
            pad_id = self.tokenizer.eos_token_id
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in features:
            n = max_len - len(x["input_ids"])
            batch["input_ids"].append(x["input_ids"] + [pad_id] * n)
            batch["attention_mask"].append(x["attention_mask"] + [0] * n)
            batch["labels"].append(x["labels"] + [-100] * n)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

def train_lora(
    model_id: str = "google/gemma-3-1b-it",
    dataset_path: str = "data/medpulse_synthetic.jsonl",
    output_dir: str = "outputs/medpulse_lora",
    max_length: int = 1024,
    epochs: int = 2,
):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from datasets import Dataset

    if not torch.cuda.is_available():
        raise RuntimeError("A CUDA GPU is required for this 4-bit starter. In Colab choose Runtime > Change runtime type > GPU.")
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id, token=True, quantization_config=quant_config,
        device_map="auto", torch_dtype=torch.float16,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    records = load_records(dataset_path)
    tokenized = [format_example(tokenizer, row, max_length=max_length) for row in records]
    train_dataset = Dataset.from_list(tokenized)
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=1,
        save_strategy="epoch",
        save_total_limit=2,
        fp16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        report_to="none",
        remove_unused_columns=False,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_dataset, data_collator=PadCollator(tokenizer))
    trainer.train()
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"LoRA adapter saved to: {output_dir}")
    return model, tokenizer

def load_adapter_for_inference(model_id="google/gemma-3-1b-it", adapter_dir="outputs/medpulse_lora"):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel
    if not torch.cuda.is_available():
        raise RuntimeError("GPU is recommended for this demo.")
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, token=True)
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    base = AutoModelForCausalLM.from_pretrained(model_id, token=True, quantization_config=quant_config, device_map="auto", torch_dtype=torch.float16)
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    return model, tokenizer

def generate_summary(model, tokenizer, discharge_text: str, max_new_tokens: int = 220) -> str:
    import torch
    prompt = (
        "You are MedPulse, an educational assistant that rewrites discharge summaries "
        "in plain language. Preserve medicine names/doses, follow-up instructions, warning "
        "signs, and emergency directions. Do not add advice. If information is missing, "
        "tell the reader to check with the care team.\n\n"
        f"Discharge summary: {discharge_text}\nPlain-language summary:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()


In [ ]:
import inspect
from transformers.generation.utils import GenerationMixin

print("generate() source file:")
print(inspect.getfile(GenerationMixin.generate))

print("\ngenerate() signature:")
print(inspect.signature(GenerationMixin.generate))

In [ ]:
import inspect
from transformers.generation.utils import GenerationMixin

print(inspect.getsource(GenerationMixin.generate))

In [ ]:
with torch.inference_mode():
    output = medpulse_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=base_tokenizer.pad_token_id,
        top_p=None,
        top_k=None,
    )

In [ ]:
# 4) Generate and validate the synthetic dataset
import sys
sys.path.insert(0, "/content/medpulse/src")
from medpulse_data import write_jsonl, validate_jsonl
dataset_path = write_jsonl("/content/medpulse/data/medpulse_synthetic.jsonl")
print("Dataset:", dataset_path)
print("Valid synthetic records:", validate_jsonl(dataset_path))
print(dataset_path.read_text()[:1000])


In [ ]:
# 5) Load the base model and test a baseline response before fine-tuning
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-1b-it"  # Change to google/gemma-3-4b-it only if your GPU can handle it.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, token=True, quantization_config=quant_config,
    device_map="auto", torch_dtype=torch.float16,
)
base_model.eval()
test_input = ("You were treated for dehydration. Drink fluids as advised. Follow up in 3 days. "
              "Seek urgent care if you faint or become confused.")
prompt = ("Rewrite this discharge summary in plain language. Preserve follow-up and warning signs; "
          "do not add medical advice.\n\nDischarge summary: " + test_input + "\nPlain-language summary:")
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(base_model.device)
with torch.no_grad():
    out = base_model.generate(**inputs, max_new_tokens=160, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))


In [ ]:
# 6) Fine-tune a LoRA adapter (4-bit base model; small educational dataset)
# This cell can take several minutes. Training output is saved under /content/medpulse/outputs.
from medpulse_train import train_lora
del base_model, tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

trained_model, trained_tokenizer = train_lora(
    model_id=MODEL_ID,
    dataset_path="/content/medpulse/data/medpulse_synthetic.jsonl",
    output_dir="/content/medpulse/outputs/medpulse_lora",
    max_length=1024,
    epochs=2,
)


In [ ]:

# Diagnose the MedPulse generation function directly

from medpulse_train import generate_summary

test_input = (
    "You were treated for dehydration. Drink fluids regularly as "
    "advised by your care team. Take medicines as prescribed. "
    "Follow up with your clinician in 3 days. Seek urgent care "
    "if you faint, become confused, cannot keep fluids down, "
    "or feel rapidly worse."
)

result = generate_summary(
    trained_model,
    trained_tokenizer,
    test_input
)

print("Result type:", type(result))
print("Result representation:", repr(result))
print("Result length:", len(result) if isinstance(result, str) else "Not a string")


In [ ]:

import torch

def test_generate_summary(model, tokenizer, text):
    prompt = (
        "Rewrite the following fictional discharge summary in clear, "
        "simple language. Preserve the original medical instructions. "
        "Do not invent diagnoses, medicines, or treatments.\n\n"
        f"Discharge summary:\n{text}\n\n"
        "Plain-language summary:"
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens, not the input prompt.
    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0, prompt_length:]

    result = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    if not result:
        return (
            "No new text was generated. Check the model output, "
            "generation configuration, and model compatibility."
        )

    return result


test_result = test_generate_summary(
    trained_model,
    trained_tokenizer,
    test_input
)

print(repr(test_result))


In [ ]:

import torch

print("Model class:", type(trained_model).__name__)
print("Tokenizer class:", type(trained_tokenizer).__name__)
print("Device:", next(trained_model.parameters()).device)
print("EOS token ID:", trained_tokenizer.eos_token_id)
print("PAD token ID:", trained_tokenizer.pad_token_id)
print("Model config:", trained_model.config.model_type)
print("Generation config:", trained_model.generation_config)

prompt = (
    "Rewrite this fictional discharge summary in simple language. "
    "Do not add medical advice.\n"
    "Discharge summary: You were treated for dehydration. "
    "Follow up with your clinician in 3 days.\n"
    "Plain-language summary:"
)

inputs = trained_tokenizer(prompt, return_tensors="pt")
device = next(trained_model.parameters()).device
inputs = {key: value.to(device) for key, value in inputs.items()}

print("Input token count:", inputs["input_ids"].shape[1])

with torch.inference_mode():
    generated = trained_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        min_new_tokens=5,
        pad_token_id=(
            trained_tokenizer.pad_token_id
            if trained_tokenizer.pad_token_id is not None
            else trained_tokenizer.eos_token_id
        ),
    )

print("Generated tensor shape:", tuple(generated.shape))
print("Input length:", inputs["input_ids"].shape[1])
print("Generated token IDs:", generated[0].tolist())

new_tokens = generated[0, inputs["input_ids"].shape[1]:]
print("New token count:", new_tokens.numel())
print("New token IDs:", new_tokens.tolist())
print(
    "Decoded new tokens:",
    repr(trained_tokenizer.decode(
        new_tokens, skip_special_tokens=False
    ))
)


In [ ]:

import inspect
import medpulse_train

print(inspect.getsource(medpulse_train.generate_summary))


In [ ]:

print("Model class:", type(trained_model))
print("Is PEFT model:", hasattr(trained_model, "peft_config"))
print("Training output directory:")
print("/content/medpulse/outputs/medpulse_lora")

print("\nEOS token:", repr(trained_tokenizer.eos_token))
print("PAD token:", repr(trained_tokenizer.pad_token))

print("\nGeneration defaults:")
print(trained_model.generation_config.to_dict())


In [ ]:

import torch

prompt = (
    "You are MedPulse, an educational assistant that rewrites "
    "discharge summaries in plain language. Do not add medical advice.\n\n"
    "Discharge summary: You were treated for dehydration. "
    "Follow up with your clinician in 3 days.\n"
    "Plain-language summary:"
)

inputs = trained_tokenizer(
    prompt,
    return_tensors="pt"
).to(next(trained_model.parameters()).device)


def run_generation(model, label):
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.05,
            eos_token_id=trained_tokenizer.eos_token_id,
            pad_token_id=trained_tokenizer.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output[0, prompt_len:]

    print(f"\n--- {label} ---")
    print("Token count:", new_tokens.numel())
    print(
        "Generated token IDs:",
        new_tokens[:30].tolist()
    )
    print(
        "Decoded output:",
        repr(trained_tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ))
    )


# Test the trained adapter first.
run_generation(trained_model, "LoRA adapter enabled")

# Temporarily disable the adapter for comparison.
with trained_model.disable_adapter():
    run_generation(trained_model, "LoRA adapter disabled")


In [ ]:

import json
from pathlib import Path

dataset_path = Path(
    "/content/medpulse/data/medpulse_synthetic.jsonl"
)

print("Dataset exists:", dataset_path.exists())

if dataset_path.exists():
    records = []

    with dataset_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    print("Number of records:", len(records))
    print("First two records:")

    for record in records[:2]:
        print(json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ))


In [ ]:

import torch

print("Model name/path:", trained_model.config._name_or_path)
print("Tokenizer name/path:", trained_tokenizer.name_or_path)
print("Chat template available:", bool(trained_tokenizer.chat_template))

messages = [
    {
        "role": "user",
        "content": (
            "Rewrite this fictional discharge summary in simple "
            "language. Preserve the instructions and do not add "
            "medical advice.\n\n"
            "Discharge summary: You were treated for dehydration. "
            "Follow up with your clinician in 3 days.\n\n"
            "Return only the plain-language summary."
        ),
    }
]

if not trained_tokenizer.chat_template:
    print("No chat template is configured for this tokenizer.")
else:
    formatted_prompt = trained_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    print("\nFormatted prompt:")
    print(formatted_prompt)

    inputs = trained_tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(next(trained_model.parameters()).device)

    def test_chat_generation(model, label):
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=trained_tokenizer.pad_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]
        new_tokens = output[0, prompt_len:]

        print(f"\n--- {label} ---")
        print("New token count:", new_tokens.numel())
        print(
            "Decoded output:",
            repr(trained_tokenizer.decode(
                new_tokens,
                skip_special_tokens=True,
            )),
        )

    test_chat_generation(trained_model, "Adapter enabled")

    with trained_model.disable_adapter():
        test_chat_generation(trained_model, "Adapter disabled")


In [ ]:
from pathlib import Path

adapter_dir = Path("/content/medpulse/outputs/medpulse_lora")

print("Adapter directory exists:", adapter_dir.exists())

if adapter_dir.exists():
    for file in adapter_dir.iterdir():
        print(file.name)

In [ ]:

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-3-1b-it"

# Release the current PEFT model from memory.
del trained_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load the original tokenizer and base model afresh.
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager",
)

print("Fresh model:", type(base_model).__name__)
print("Model device:", next(base_model.parameters()).device)
print("Tokenizer:", base_tokenizer.name_or_path)


## Step 3 — Test the fresh base model

messages = [
    {
        "role": "user",
        "content": (
            "Rewrite this fictional discharge summary in simple "
            "language without adding medical advice.\n\n"
            "You were treated for dehydration. "
            "Follow up with your clinician in 3 days."
        ),
    }
]

prompt = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = base_tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(next(base_model.parameters()).device)

with torch.inference_mode():
    output = base_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=base_tokenizer.pad_token_id,
    )

new_tokens = output[0, inputs["input_ids"].shape[1]:]

print("Generated token IDs:", new_tokens[:30].tolist())
print(
    "Fresh base-model output:",
    repr(base_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )),
)


In [ ]:

messages = [
    {
        "role": "user",
        "content": (
            "Rewrite the source in simple language.\n"
            "Rules:\n"
            "1. Preserve every medical fact and instruction.\n"
            "2. Preserve the exact follow-up timing.\n"
            "3. Do not add advice, warnings, explanations, headings, "
            "or facts absent from the source.\n"
            "4. Return only the rewritten text.\n\n"
            "Source: You were treated for dehydration. "
            "Follow up with your clinician in 3 days."
        ),
    }
]

prompt = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = base_tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(next(medpulse_model.parameters()).device)

with torch.inference_mode():
    output = medpulse_model.generate(
        **inputs,
        max_new_tokens=70,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=base_tokenizer.pad_token_id,
        top_p=None,
        top_k=None,
    )

new_tokens = output[0, inputs["input_ids"].shape[1]:]

strict_result = base_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
).strip()

print("Strict evaluation output:")
print(strict_result)


In [ ]:

import torch
from peft import PeftModel

ADAPTER_DIR = "/content/medpulse/outputs/medpulse_lora"

# Load the saved adapter onto the verified base model.
medpulse_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)

medpulse_model.eval()

messages = [
    {
        "role": "user",
        "content": (
            "Rewrite this fictional discharge summary in simple "
            "language. Preserve all instructions. Do not add "
            "medical advice or information that is not in the source.\n\n"
            "Discharge summary: You were treated for dehydration. "
            "Follow up with your clinician in 3 days."
        ),
    }
]

prompt = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = base_tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(next(medpulse_model.parameters()).device)

with torch.inference_mode():
    output = medpulse_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=base_tokenizer.pad_token_id,
    )

new_tokens = output[0, inputs["input_ids"].shape[1]:]

adapter_result = base_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
).strip()

print("Saved adapter output:")
print(repr(adapter_result))


In [ ]:
import torch
import peft
import transformers
import importlib.metadata as metadata

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)
print("TorchAO:", metadata.version("torchao"))
print("CUDA available:", torch.cuda.is_available())

In [ ]:
%pip uninstall -y torchao

In [ ]:

import torch
from peft import PeftModel
from pathlib import Path

ADAPTER_DIR = "/content/medpulse/outputs/medpulse_lora"

# Verify that the saved adapter exists.
adapter_path = Path(ADAPTER_DIR)
assert adapter_path.is_dir(), f"Adapter directory not found: {ADAPTER_DIR}"
assert (adapter_path / "adapter_config.json").exists(), (
    "adapter_config.json is missing. Check the saved adapter directory."
)

# Confirm that the base model and tokenizer are already loaded.
assert base_model is not None, "Load base_model before this cell."
assert base_tokenizer is not None, "Load base_tokenizer before this cell."

# Attach the saved LoRA adapter.
medpulse_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)

medpulse_model.eval()

messages = [
    {
        "role": "user",
        "content": (
            "Rewrite this fictional discharge summary in simple "
            "language. Preserve all instructions. Do not add "
            "medical advice or information that is not in the source.\n\n"
            "Discharge summary: You were treated for dehydration. "
            "Follow up with your clinician in 3 days."
        ),
    }
]

prompt = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = base_tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(next(medpulse_model.parameters()).device)

with torch.inference_mode():
    output = medpulse_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=base_tokenizer.pad_token_id,
        top_p=None,
        top_k=None,
    )

new_tokens = output[0, inputs["input_ids"].shape[1]:]

adapter_result = base_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
).strip()

print("Saved adapter output:")
print(repr(adapter_result))


In [ ]:

messages = [
    {
        "role": "user",
        "content": (
            "Rewrite the source in simple language.\n"
            "Rules:\n"
            "1. Preserve every medical fact and instruction.\n"
            "2. Preserve the exact follow-up timing.\n"
            "3. Do not add advice, warnings, explanations, headings, "
            "or facts absent from the source.\n"
            "4. Return only the rewritten text.\n\n"
            "Source: You were treated for dehydration. "
            "Follow up with your clinician in 3 days."
        ),
    }
]

prompt = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = base_tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(next(medpulse_model.parameters()).device)

with torch.inference_mode():
    output = medpulse_model.generate(
        **inputs,
        max_new_tokens=70,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=base_tokenizer.pad_token_id,
        top_p=None,
        top_k=None,
    )

new_tokens = output[0, inputs["input_ids"].shape[1]:]

strict_result = base_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
).strip()

print("Strict evaluation output:")
print(strict_result)


In [ ]:
import os
from pathlib import Path

output_dir = Path("/content/medpulse/outputs/medpulse_lora")

print("Output directory exists:", output_dir.exists())

if output_dir.exists():
    print("\nFiles and folders created during training:")

    for path in sorted(output_dir.rglob("*")):
        if path.is_file():
            print(f"FILE: {path.relative_to(output_dir)}")
        elif path.is_dir():
            print(f"DIR:  {path.relative_to(output_dir)}")

    adapter_config = output_dir / "adapter_config.json"
    adapter_weights = (
        list(output_dir.glob("adapter_model.safetensors"))
        + list(output_dir.glob("adapter_model.bin"))
    )

    print("\nLoRA adapter config exists:", adapter_config.exists())
    print("LoRA adapter weights exist:", bool(adapter_weights))

    checkpoints = list(output_dir.glob("checkpoint-*"))
    print("Checkpoint folders:", [p.name for p in checkpoints])
else:
    print("Training output directory has not been created.")

In [ ]:

import torch
from peft import PeftModel

ADAPTER_DIR = "/content/medpulse/outputs/medpulse_lora"

# Load the saved adapter onto the verified base model.
medpulse_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)

medpulse_model.eval()

messages = [
    {
        "role": "user",
        "content": (
            "Rewrite this fictional discharge summary in simple "
            "language. Preserve all instructions. Do not add "
            "medical advice or information that is not in the source.\n\n"
            "Discharge summary: You were treated for dehydration. "
            "Follow up with your clinician in 3 days."
        ),
    }
]

prompt = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = base_tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(next(medpulse_model.parameters()).device)

with torch.inference_mode():
    output = medpulse_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=base_tokenizer.pad_token_id,
    )

new_tokens = output[0, inputs["input_ids"].shape[1]:]

adapter_result = base_tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
).strip()

print("Saved adapter output:")
print(repr(adapter_result))


In [ ]:

import torch

def medpulse_generate_summary(text, max_new_tokens=160):
    if not text or not text.strip():
        return "Please enter a fictional discharge summary."

    messages = [
        {
            "role": "user",
            "content": (
                "You are MedPulse, an educational plain-language "
                "rewriting assistant.\n"
                "Rewrite the supplied discharge summary in simple, "
                "calm language.\n"
                "Preserve all medical facts, medicine names and doses, "
                "dates, follow-up instructions, warning signs, and "
                "emergency directions exactly in meaning.\n"
                "Do not add diagnoses, treatments, reasons for follow-up, "
                "or other information absent from the source.\n"
                "Do not add headings, disclaimers, or commentary.\n"
                "Return only the rewritten summary.\n\n"
                f"Source discharge summary:\n{text}"
            ),
        }
    ]

    prompt = base_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    device = next(medpulse_model.parameters()).device
    inputs = base_tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(device)

    # Preserve the model's configured end-of-turn token IDs.
    eos_ids = medpulse_model.generation_config.eos_token_id

    with torch.inference_mode():
        output = medpulse_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=base_tokenizer.pad_token_id,
            eos_token_id=eos_ids,
        )

    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output[0, prompt_len:]

    result = base_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    if not result:
        return (
            "No summary was generated. Please check the model "
            "generation output in Colab."
        )

    return result


# Test before launching Gradio.
test_summary = medpulse_generate_summary(
    "You were treated for dehydration. Drink fluids regularly as "
    "advised by your care team. Take medicines as prescribed. "
    "Follow up with your clinician in 3 days. Seek urgent care "
    "if you faint or become confused."
)

print(test_summary)


In [ ]:

import json
from pathlib import Path

ADAPTER_DIR = "/content/medpulse/outputs/medpulse_lora"

config_path = Path(ADAPTER_DIR) / "adapter_config.json"

assert config_path.exists(), (
    f"Missing adapter configuration: {config_path}"
)

with open(config_path, "r") as f:
    adapter_config = json.load(f)

print("Base model recorded in adapter:")
print(adapter_config.get("base_model_name_or_path"))

print("\nLoRA task type:")
print(adapter_config.get("task_type"))

print("\nLoRA target modules:")
print(adapter_config.get("target_modules"))


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-1b-it"  # Change to the verified base model ID

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
)

print("Base model loaded:", MODEL_ID)
print("Tokenizer loaded:", base_tokenizer.__class__.__name__)


In [ ]:
# 7) Basic smoke test after training (not a clinical evaluation)
from medpulse_train import generate_summary
test_input = ("You were treated for dehydration. Drink fluids as advised. Follow up in 3 days. "
              "Seek urgent care if you faint or become confused.")
print(generate_summary(trained_model, trained_tokenizer, test_input))


In [ ]:
# 8) Save adapter to Google Drive so it persists after the Colab runtime ends
from google.colab import drive
drive.mount("/content/drive")
import shutil
drive_dest = Path("/content/drive/MyDrive/MedPulse/medpulse_lora")
drive_dest.parent.mkdir(parents=True, exist_ok=True)
if drive_dest.exists():
    shutil.rmtree(drive_dest)
shutil.copytree("/content/medpulse/outputs/medpulse_lora", drive_dest)
print("Adapter saved to Google Drive:", drive_dest)
print("Keep adapter private; do not commit it to a public repository.")


In [ ]:

import gradio as gr

def simplify(text):
    if not text or not text.strip():
        return "Enter a fictional example first."

    if len(text) > 6000:
        return "Please keep the input under 6,000 characters."

    try:
        return medpulse_generate_summary(text)
    except Exception as exc:
        print("MedPulse generation error:", repr(exc))
        return "Generation failed. Check the Colab output logs."


with gr.Blocks(title="MedPulse — Educational Demo") as demo:
    gr.Markdown(
        "# MedPulse\n"
        "**Educational prototype only — not medical advice.** "
        "Use synthetic data only. Clinician review is required."
    )

    source = gr.Textbox(
        label="Synthetic discharge summary",
        lines=8,
        placeholder="Enter a fictional example only."
    )

    output = gr.Textbox(
        label="Plain-language draft",
        lines=8,
        interactive=False
    )

    gr.Button("Simplify summary").click(
        fn=simplify,
        inputs=source,
        outputs=output
    )

    gr.Examples(
        examples=[[
            "You were treated for dehydration. Drink fluids regularly "
            "as advised by your care team. Take medicines as prescribed. "
            "Follow up with your clinician in 3 days. Seek urgent care "
            "if you faint, become confused, cannot keep fluids down, "
            "or feel rapidly worse."
        ]],
        inputs=source
    )

demo.launch(share=True)


In [ ]:
# 9) Launch Gradio and get a temporary public URL
# Anyone who has the URL may be able to access this demo while the runtime is active.
# Use only fictional examples. Do not paste patient records.
import gradio as gr
from medpulse_train import generate_summary

def simplify(text):
    if not text or not text.strip():
        return "Enter a fictional example first."
    if len(text) > 6000:
        return "Please keep the input under 6,000 characters."
    return generate_summary(trained_model, trained_tokenizer, text)

with gr.Blocks(title="MedPulse — Educational Demo") as demo:
    gr.Markdown("# MedPulse\n**Educational prototype only — not medical advice.** Use synthetic data only; clinician review is required.")
    source = gr.Textbox(label="Synthetic discharge summary", lines=8,
        placeholder="Paste a fictional example. Do not enter real patient information.")
    output = gr.Textbox(label="Plain-language draft", lines=8)
    gr.Button("Simplify summary").click(simplify, inputs=source, outputs=output)
    gr.Examples(examples=[[
        "You were treated for dehydration. Drink fluids regularly as advised by your care team. "
        "Take medicines as prescribed. Follow up with your clinician in 3 days. Seek urgent care "
        "if you faint, become confused, cannot keep fluids down, or feel rapidly worse."
    ]], inputs=source)
demo.launch(share=True)
